# 00 - Setup, download Imagenette, and cache VAE latents

    This notebook prepares the data for the latent DiT reproduction. It installs dependencies, downloads Imagenette, loads a pretrained Stable-Diffusion VAE, caches 32x32x4 latents for train/validation images, and saves visual inspection grids.

    Expected outputs:
    - `data/imagenette2-320/`
    - `data/imagenette_latents/train/shard_*.pt`
    - `data/imagenette_latents/val/shard_*.pt`
    - `figures/vae_reconstructions.png`


## Install dependencies

    This cell installs the Python packages required by the notebooks. It uses the project `requirements.txt` so the environment is reproducible.


In [ ]:
%pip install -q -r requirements.txt

## Import libraries and configure paths

    This cell adds the local `src/` directory to the Python path, creates output directories, and selects the available device.


In [ ]:
import sys
from pathlib import Path
import json
import torch
import matplotlib.pyplot as plt
from torchvision.utils import make_grid

PROJECT_ROOT = Path.cwd()
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from dit.data import get_imagenette_dataset, save_class_mapping, latent_stats, IMAGENETTE_URL
from dit.latent import load_vae, cache_latents, save_reconstruction_grid

DATA_DIR = PROJECT_ROOT / "data"
FIG_DIR = PROJECT_ROOT / "figures"
LATENT_DIR = DATA_DIR / "imagenette_latents"
for d in [DATA_DIR, FIG_DIR, LATENT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## Download Imagenette with curl/wget

    This cell downloads the Imagenette2-320 archive if it is not already present and extracts it into `data/`. Imagenette is an ImageNet-derived 10-class subset that makes the DiT application feasible.


In [ ]:
import subprocess, os

archive = DATA_DIR / "imagenette2-320.tgz"
extracted = DATA_DIR / "imagenette2-320"
if not extracted.exists():
    if not archive.exists():
        print("Downloading Imagenette...")
        cmd = f"curl -L {IMAGENETTE_URL} -o {archive}"
        subprocess.check_call(cmd, shell=True)
    print("Extracting Imagenette...")
    subprocess.check_call(f"tar -xzf {archive} -C {DATA_DIR}", shell=True)
else:
    print("Imagenette already exists:", extracted)

print("Train folders:", len(list((extracted / "train").iterdir())))
print("Val folders:", len(list((extracted / "val").iterdir())))

## Load Imagenette and inspect real images

    This cell builds ImageFolder datasets with the exact preprocessing used by the VAE: resize/crop to 256x256, convert to tensor, and normalize to [-1, 1]. It also saves the class mapping.


In [ ]:
train_ds = get_imagenette_dataset(DATA_DIR, split="train", image_size=256)
val_ds = get_imagenette_dataset(DATA_DIR, split="val", image_size=256)
class_mapping = save_class_mapping(train_ds, DATA_DIR / "imagenette_class_to_idx.json")
print("Train images:", len(train_ds))
print("Val images:", len(val_ds))
print("Classes:", class_mapping)

n = 16
imgs = torch.stack([train_ds[i][0] for i in range(n)])
imgs_vis = (imgs.clamp(-1, 1) + 1) / 2
grid = make_grid(imgs_vis, nrow=4)
plt.figure(figsize=(6, 6))
plt.imshow(grid.permute(1, 2, 0))
plt.axis("off")
plt.title("Imagenette training images")
plt.show()


## Load the pretrained VAE

    The DiT paper trains diffusion in a pretrained VAE latent space. This cell loads `stabilityai/sd-vae-ft-mse`, freezes it, and prepares it for latent caching.


In [ ]:
vae = load_vae(device)
print("Loaded VAE and froze parameters.")

## Visual VAE reconstruction check

    This cell encodes a small batch of real images into VAE latents and decodes them back to pixels. The saved grid contains originals followed by reconstructions.


In [ ]:
recon_path = FIG_DIR / "vae_reconstructions.png"
save_reconstruction_grid(vae, val_ds, recon_path, device=device, n=16)
print("Saved:", recon_path)
from PIL import Image
import matplotlib.pyplot as plt
img = Image.open(recon_path)
plt.figure(figsize=(10, 5))
plt.imshow(img)
plt.axis("off")
plt.title("Top rows: originals; bottom rows: VAE reconstructions")
plt.show()

## Cache train and validation latents

    This is the main preprocessing step. It saves VAE latents as shard files so training does not need to run the VAE encoder every iteration.


In [ ]:
OVERWRITE_LATENTS = False
CACHE_BATCH_SIZE = 32 if device.type == "cuda" else 4
NUM_WORKERS = 4

cache_latents(
    vae, train_ds, LATENT_DIR / "train",
    batch_size=CACHE_BATCH_SIZE, num_workers=NUM_WORKERS,
    device=device, overwrite=OVERWRITE_LATENTS,
)
cache_latents(
    vae, val_ds, LATENT_DIR / "val",
    batch_size=CACHE_BATCH_SIZE, num_workers=NUM_WORKERS,
    device=device, overwrite=OVERWRITE_LATENTS,
)

## Check cached latent statistics

    This cell verifies that the cached tensors have the expected DiT input shape: 4 channels and 32x32 spatial resolution for 256x256 images.


In [ ]:
train_stats = latent_stats(LATENT_DIR / "train")
val_stats = latent_stats(LATENT_DIR / "val")
print("Train latent stats:", train_stats)
print("Val latent stats:", val_stats)
assert train_stats["latent_shape"] == (4, 32, 32), train_stats
assert val_stats["latent_shape"] == (4, 32, 32), val_stats
